In [4]:
import os
import subprocess
import io

PWD = "mysecret" 
GRAPH = "http://ns.inria.fr/movida/graph"

#Load the data in the graph
def importData():
    for item in os.listdir("../DataLifting_output"):
        try:
            print(f"Loading {item}...")
            
            requete_sql = f"ld_dir('/usr/share/proj', '{item}', 'http://ns.inria.fr/movida/graph'); rdf_loader_run();"
            
            commande = [
                "docker", "exec", "virtuoso-server", 
                "isql", "-U", "dba", "-P", "mysecret", 
                f"exec={requete_sql}"
            ]
            
            resultat = subprocess.run(commande, capture_output=True, text=True)
        
            if resultat.returncode == 0:
                print(f"{item} loaded !")
            else:
                print(f"Error while loading {item} file :")
                print(resultat.stderr)
        except OSError as e:
            print(f"Error:{ e.strerror}")


#delete the graph
def resetGraph():
    print(f"Cleaning the graph...")
    
    requete = f"""
    CHECKPOINT;
    SPARQL CLEAR GRAPH <{GRAPH}>;
    DELETE FROM DB.DBA.load_list;
    exit;
    """
    
    commande = ["docker", "exec", "-i", "virtuoso-server", "isql", "-U", "dba", "-P", PWD]
    
    try:
        res = subprocess.run(commande, input=requete, capture_output=True, text=True, timeout=20)
        
        if "Error" not in res.stdout:
            print("Graph cleaned !")
            importData()
        else:
            print("Error :", res.stdout)
            
    except subprocess.TimeoutExpired:
        print("Timeout : Virtuoso didn't respond in time... :(")

resetGraph()


Cleaning the graph...
Graph cleaned !
Loading indicator.ttl...
indicator.ttl loaded !
Loading record.ttl...
record.ttl loaded !
Loading record1.ttl...
record1.ttl loaded !
Loading record2.ttl...
record2.ttl loaded !
Loading record3.ttl...
record3.ttl loaded !
Loading record4.ttl...
record4.ttl loaded !
Loading record5.ttl...
record5.ttl loaded !
Loading record6.ttl...
record6.ttl loaded !
Loading record7.ttl...
record7.ttl loaded !
Loading record8.ttl...
record8.ttl loaded !
Loading space.ttl...
space.ttl loaded !


In [5]:
prefixes = {
    "http://ns.inria.fr/movida/ontology#": "mvdo:",
    "http://ns.inria.fr/movida/thesaurus#": "mvdth:",
    "http://ns.inria.fr/movida/data#": ":",
    "http://www.w3.org/1999/02/22-rdf-syntax-ns#": "rdf:",
    "http://www.w3.org/2000/01/rdf-schema#": "rdfs:",
    "http://www.w3.org/2002/07/owl#": "owl:",
    "http://www.w3.org/2001/XMLSchema#": "xsd:",
    "http://www.w3.org/2004/02/skos/core#": "skos:",
    "http://www.w3.org/2006/time#": "time:",
    "http://www.opengis.net/ont/geosparql#": "geo:",
    "http://www.w3.org/ns/prov#": "prov:",
    "http://qudt.org/schema/qudt/": "qudt:",
    "https://qudt.org/3.3.0/vocab/quantitykind": "quantitykind:",
    "http://www.w3.org/ns/sosa/": "sosa:"
}

def shortenURI(uri):
    if not isinstance(uri, str):
        return uri
    for completeURI, prefix in prefixes.items():
        if uri.startswith(completeURI):
            return uri.replace(completeURI, prefix)
    return uri

In [6]:
queryPrefixes ="""
PREFIX :        <http://ns.inria.fr/movida/data#>
PREFIX qudt:    <http://qudt.org/schema/qudt/>
PREFIX owl:     <http://www.w3.org/2002/07/owl#>
PREFIX mvdo:    <http://ns.inria.fr/movida/ontology#>
PREFIX xsd:     <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:    <http://www.w3.org/2004/02/skos/core#>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX geo:     <http://www.opengis.net/ont/geosparql#>
PREFIX quantitykind:  <https://qudt.org/3.3.0/vocab/quantitykind>
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX mvdth:   <http://ns.inria.fr/movida/thesaurus#>
PREFIX time:    <http://www.w3.org/2006/time#>
PREFIX prov:    <http://www.w3.org/ns/prov#>
PREFIX sosa:    <http://www.w3.org/ns/sosa/>
"""

In [7]:
import pandas as pd
import requests

def executeSparqlQuery(sparqlQuery, url="http://localhost:8890/sparql"):
    sparqlQuery = queryPrefixes + sparqlQuery
    parameters = {
        "query": sparqlQuery,
        "format": "text/csv"
    }
    try:
        answer = requests.get(url, params=parameters)
        answer.raise_for_status()

        df = pd.read_csv(io.StringIO(answer.text))

        for colonne in df.columns:
            df[colonne] = df[colonne].apply(shortenURI)

        return df
    except Exception as e:
        print(f"Error : {e}")

    return pd.DataFrame()



# Structural Queries

number of triples

In [8]:
query = """
    SELECT (COUNT(*) AS ?nbTriples)
    FROM <http://ns.inria.fr/movida/graph>
    WHERE {
      ?s ?p ?o .
    }
"""

df = executeSparqlQuery(query)
display(df)

,nbTriples
0,5090403


number of mvdo:Territory

In [9]:
query = """
    SELECT (COUNT(*) AS ?nbTerritory)
    WHERE {
      ?territory rdf:type mvdo:Territory .
    }
"""

df = executeSparqlQuery(query)
display(df)

,nbTerritory
0,1


number of mvdo:TerritorialPartition

In [10]:
query = """
    SELECT (COUNT(*) AS ?nbTerritorialPartition)
    WHERE {
      ?partition rdf:type mvdo:TerritorialPartition .
    }
"""

df = executeSparqlQuery(query)
display(df)

,nbTerritorialPartition
0,3


number of mvdo:SpatialZone

In [11]:
query = """
    SELECT (COUNT(*) AS ?nbSpatialZone)
    WHERE {
      ?zone rdf:type mvdo:SpatialZone .
    }
"""

df = executeSparqlQuery(query)
display(df)

,nbSpatialZone
0,147


1 spatial zone is missing, 148 are exprected.<br>
Reason : "code": 1.0, "name": "Grenoble" (same name and ID) is defined in both D10 and D30 territorial parition


number of mvdo:Indicator

In [12]:
query = """
    SELECT (COUNT(*) AS ?nbIndicator)
    WHERE {
      ?indicator rdf:type ?o .
      VALUES ?o { mvdo:Indicator mvdo:TerritoryIndicator mvdo:MovementIndicator }
    }
"""

df = executeSparqlQuery(query)
display(df)

,nbIndicator
0,15


number of mvdo:DataRecord

In [13]:
query = """
    SELECT (COUNT(*) AS ?nbDataRecord)
    WHERE {
      ?record rdf:type mvdo:DataRecord .
    }
"""

df = executeSparqlQuery(query)
display(df)

,nbDataRecord
0,877887


6 575 data records are missing.<br>
Reason : space duplication caused some record IDs to merge.<br>
These 6 575 missing data records correspond to the number of records defined on D30 partition. 

number of mvdo:DataRecord linked to a mvdo:Presence indicator

In [14]:
query = """
    SELECT (COUNT(*) AS ?nbDataRecord)
    WHERE {
      ?record rdf:type mvdo:DataRecord .
      ?record mvdo:isRecordOf ?indicator .
      ?indicator rdf:type mvdo:TerritoryIndicator .
      ?indicator mvdo:hasIndicatorType mvdth:Presence
    }
"""

df = executeSparqlQuery(query)
display(df)

,nbDataRecord
0,870096


6 522 data records are missing.<br>
Reason : space duplication caused some record IDs to merge.<br>
These 6 522 missing data records correspond to the number of records defined on D30 partition. 

number of mvdo:DataRecord linked to a mvdo:Fluctuation indicator

In [15]:
query = """
    SELECT (COUNT(*) AS ?nbDataRecord)
    WHERE {
      ?record rdf:type mvdo:DataRecord .
      ?record mvdo:isRecordOf ?indicator .
      ?indicator rdf:type mvdo:TerritoryIndicator .
      ?indicator mvdo:hasIndicatorType mvdth:Fluctuation
    }
"""

df = executeSparqlQuery(query)
display(df)

,nbDataRecord
0,7644


52 data records are missing.<br>
Reason : space duplication caused some record IDs to merge.<br>
These 52 missing data records correspond to the number of records defined on D30 partition. 

number of mvdo:DataRecord linked to a mvdo:Attractiveness indicator

In [16]:
query = """
    SELECT (COUNT(*) AS ?nbDataRecord)
    WHERE {
      ?record rdf:type mvdo:DataRecord .
      ?record mvdo:isRecordOf ?indicator .
      ?indicator rdf:type mvdo:TerritoryIndicator .
      ?indicator mvdo:hasIndicatorType mvdth:Attractiveness
    }
"""

df = executeSparqlQuery(query)
display(df)

,nbDataRecord
0,147


1 data record is missing.<br>
Reason : space duplication caused some record IDs to merge.<br>
This missing data record correspond to the number of records defined on D30 partition. 

Every indicator with the property mvdo:hasTheme linked to mvdth:TransportationMode are linked to records with the property mvdo:hasTransportationMode

In [17]:
query = """
    SELECT DISTINCT ?indicator
    WHERE {
      ?indicator rdf:type ?indicatorType .
      VALUES ?indicatorType { mvdo:Indicator mvdo:TerritoryIndicator mvdo:MovementIndicator }
      ?indicator mvdo:hasTheme mvdth:TransportationMode .

      FILTER NOT EXISTS {
        ?record mvdo:isRecordOf ?indicator .
        ?record rdf:type mvdo:DataRecord .
        ?record mvdo:hasTransportationMode ?value .
      }
    }
"""

df = executeSparqlQuery(query)
display(df)

,indicator


Every indicator with the property mvdo:hasTheme linked to mvdth:TripPurpose are linked to records with the property mvdo:hasTripPurpose

In [18]:
query = """
    SELECT DISTINCT ?indicator
    WHERE {
      ?indicator rdf:type ?indicatorType .
      VALUES ?indicatorType { mvdo:Indicator mvdo:TerritoryIndicator mvdo:MovementIndicator }
      ?indicator mvdo:hasTheme mvdth:TripPurpose .

      FILTER NOT EXISTS {
        ?record mvdo:isRecordOf ?indicator .
        ?record rdf:type mvdo:DataRecord .
        ?record mvdo:hasTripPurpose ?value .
      }
    }
"""

df = executeSparqlQuery(query)
display(df)

,indicator


Every space has a name and an ID

In [19]:
query = """
    SELECT DISTINCT ?space
    WHERE {
      ?space rdf:type ?spaceType .
      VALUES ?spaceType { mvdo:SpatialZone mvdo:Territory }

      FILTER NOT EXISTS {
        ?space mvdo:hasName ?nameValue .
        ?space mvdo:hasID ?idValue .
      }
    }
"""

df = executeSparqlQuery(query)
display(df)

,space


# Functional Queries

### first table : Object, Space, Attribute

What is the value of the indicator I at spatial location s regarding the object o and the attribute a?

Query : What is the value of the indicator mvdth:Presence at spatial location Centre ville-101 regarding the object Movers and the attribute Car?

In [20]:
query = """
    SELECT ?record ?value
    WHERE {
      ?indicator rdf:type mvdo:TerritoryIndicator .
      ?indicator mvdo:hasIndicatorType mvdth:Presence .
      ?indicator mvdo:hasTheme mvdth:TransportationMode .
      ?indicator mvdo:hasObject mvdth:Movers .

      ?indicator mvdo:hasTerritory ?territory .
      ?territory mvdo:hasName "grenoble"^^rdfs:Literal .
      ?territory mvdo:hasID 0 .
      
      ?record mvdo:isRecordOf ?indicator .
      ?record sosa:hasSimpleResult ?value .
      ?record mvdo:hasTransportationMode mvdth:Car .

      ?record mvdo:hasSpatialLocation ?spatialZone .
      ?spatialZone mvdo:hasName "Centre ville"^^rdfs:Literal .
      ?spatialZone mvdo:hasID 101 .
    }
"""

df = executeSparqlQuery(query)
display(df)


,record,value
0,:record-082d9c5bc2525a59d318e5d1ac301ed833140e...,1582.486572
1,:record-085e628ec750dea9472db344223a0467880d2e...,1955.324951
2,:record-0a0a62150df150caac1c75b014fdc73c56487b...,1078.486938
3,:record-12b3f3b04effd3550ca9d1be7d45461f115fed...,996.750732
4,:record-16f9a4854cfdacec297b0390a4d879272ebd3c...,961.743103
...,...,...
424,:record-f1a0e08e9489e5d2f4c9da9537e7a7412684e1...,305.817993
425,:record-fba032ca6f3aed843973db00d43c2b7dffaace...,432.327515
426,:record-fc867cecaeb502f252232a6dc2e08e9f598e92...,62.716999
427,:record-fe75c923193e9fb8f4fa6c2050383b6dff5ba1...,73.086998


Query : What is the value of the indicator mvdth:Presence at spatial location Grenoble-2 regarding the object Trips and the attribute Leisure?

In [21]:
query = """
    SELECT ?record ?value
    WHERE {
      ?indicator rdf:type mvdo:TerritoryIndicator .
      ?indicator mvdo:hasIndicatorType mvdth:Presence .
      ?indicator mvdo:hasTheme mvdth:TripPurpose .
      ?indicator mvdo:hasObject mvdth:Trips .

      ?indicator mvdo:hasTerritory ?territory .
      ?territory mvdo:hasName "grenoble"^^rdfs:Literal .
      ?territory mvdo:hasID 0 .
      
      ?record mvdo:isRecordOf ?indicator .
      ?record sosa:hasSimpleResult ?value .
      ?record mvdo:hasTripPurpose mvdth:Leisure .
      
      ?record mvdo:hasSpatialLocation ?spatialZone .
      ?spatialZone mvdo:hasName "Grenoble"^^rdfs:Literal .
      ?spatialZone mvdo:hasID 2 .
    }
"""

df = executeSparqlQuery(query)
display(df)


,record,value
0,:record-33736427325769533aa2ab960bfd2eff48992b...,0.022467
1,:record-423824bb39f51991c7b7483b92f0630f43527f...,0.010904
2,:record-5d37b5fd8af6cd169fa429cae244471e219d72...,0.052585
3,:record-7cc79610c5151b660dbdb9f1ba3fad4cd3b60a...,0.027535
4,:record-98b1001ad7fc6dc8ab049135c41361a42e7c85...,0.046400
...,...,...
541,:record-ea6ec91493e3d153de777d6d9885d7e89ae732...,0.000000
542,:record-eb472346c09ef0aa13ae7d8de62f113c015937...,182.272003
543,:record-f229067f11542b0c355de81b0ae0160f92ab3e...,7655.826172
544,:record-f5e599b10801d5ae5c86ff0d9a0c81df3d1032...,133.585999


### second table : Object, Time, Attribute

What is the value of the indicator I at time unit t regarding object o and attribute a?

Query : What is the value of the indicator mvdth:Presence at the time 8-9 regarding the object Movers and the attribute Walk?

In [ ]:
query = """
    SELECT ?record ?value
    WHERE {
      ?indicator rdf:type mvdo:TerritoryIndicator .
      ?indicator mvdo:hasIndicatorType mvdth:Presence .
      ?indicator mvdo:hasTheme mvdth:TransportationMode .
      ?indicator mvdo:hasObject mvdth:Movers .

      ?indicator mvdo:hasTerritory ?territory .
      ?territory mvdo:hasName "grenoble"^^rdfs:Literal .
      ?territory mvdo:hasID 0 .
      
      ?record mvdo:isRecordOf ?indicator .
      ?record sosa:hasSimpleResult ?value .
      ?record mvdo:hasTransportationMode mvdth:Walk .
      
      ?record mvdo:hasTime ?time .

      ?time time:hasBeginning ?beginning .
      ?beginning time:inDateTime ?timeDescriptionBeginning .
      ?timeDescriptionBeginning time:hour "8"^^xsd:nonNegativeInteger .
      ?timeDescriptionBeginning time:unitType time:unitHour .

      ?time time:hasEnd ?end .
      ?end time:inDateTime ?timeDescriptionEnd .
      ?timeDescriptionEnd time:hour "9"^^xsd:nonNegativeInteger .
      ?timeDescriptionEnd time:unitType time:unitHour .
    }
"""

df = executeSparqlQuery(query)
display(df)


,record,value
0,:record-011427163883d882240d5c60b9f07cb2633977...,280.239990
1,:record-016f2a749578b9ca955db315c78da8f9656a0f...,3892.888916
2,:record-04f9c7f1d01388be9adf7178a4b77af16d112f...,862.868530
3,:record-17a1a63dffd5b6e05851a3e78f9a2fb9084535...,0.000000
4,:record-1a5d5f8776fda78bea9c5e7ca8800cec509376...,0.000000
...,...,...
2767,:record-f79d8f17364e38974b1c2685b613fa5b8ee275...,51.815388
2768,:record-fbdae88db2fef38b05ddc422725ad85307fd8d...,2.632919
2769,:record-fd3b8e1f058dac382536268acced6f167da86a...,16.725290
2770,:record-fd4aa82822a6558a7d5a62d108dd31e3c622eb...,2.811792


Query : What is the value of the indicator mvdth:Presence at the time 27-28 regarding the object Trips and the attribute Home?

In [58]:
query = """
    SELECT ?record ?value
    WHERE {
      ?indicator rdf:type mvdo:TerritoryIndicator .
      ?indicator mvdo:hasIndicatorType mvdth:Presence .
      ?indicator mvdo:hasTheme mvdth:TripPurpose .
      ?indicator mvdo:hasObject mvdth:Trips .

      ?indicator mvdo:hasTerritory ?territory .
      ?territory mvdo:hasName "grenoble"^^rdfs:Literal .
      ?territory mvdo:hasID 0 .
      
      ?record mvdo:isRecordOf ?indicator .
      ?record sosa:hasSimpleResult ?value .
      ?record mvdo:hasTripPurpose mvdth:Home .
      
      ?record mvdo:hasTime ?time .

      ?time time:hasBeginning ?beginning .
      ?beginning time:inDateTime ?timeDescriptionBeginning .
      ?timeDescriptionBeginning time:hour "27"^^xsd:nonNegativeInteger .
      ?timeDescriptionBeginning time:unitType time:unitHour .

      ?time time:hasEnd ?end .
      ?end time:inDateTime ?timeDescriptionEnd .
      ?timeDescriptionEnd time:hour "28"^^xsd:nonNegativeInteger .
      ?timeDescriptionEnd time:unitType time:unitHour .
    }
"""

df = executeSparqlQuery(query)
display(df)



,record,value
0,:record-069f5432029407e656060c48bc629a238682f1...,1620.620850
1,:record-0886c77a009e92503182f72b9f0d8b85f7e1c7...,1496.394531
2,:record-121cc76c63e81e299c5f61bcd2ff648eea37ff...,42.398624
3,:record-17b06d4bb2454da6b95fb3c323f4a1b2ebfbf7...,253.210098
4,:record-2749bc694356b5be988c0ab83d588ca718fa88...,224.921463
...,...,...
3085,:record-f520e67114e69b9bf649acff8950d417906975...,126.963997
3086,:record-f73c3d61a769b0c6790e58da4b0f596c61b85a...,841.163025
3087,:record-f75ed6a5bbbe70f501e5323b37464f580225c2...,1093.896973
3088,:record-fc767cae13e12cdb18684a392cb1e1b8ca7d2f...,3731.247070
